# ============================================================
# NOTEBOOK: 03_feature_engineering.ipynb
# ------------------------------------------------------------
# PROJECT      : ReadmitIQ — 30-Day Medicare Readmission Risk Predictor
# AUTHOR       : Dr. Nikki
# CREATED      : 2026-04-08
# LAST UPDATED : 2026-04-08
#
# PURPOSE
# -------
# Build the full, production-ready feature set for the model.
# This notebook:
#   1. Derives the 30-day readmission target variable
#   2. Calculates all 14 model features from raw claims data
#   3. Joins beneficiary demographics to inpatient claims
#   4. Applies a temporal train/test split (2008 train, 2009 test)
#   5. Saves the final feature matrices ready for modeling
#
# INPUTS
# ------
# data/raw/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv
# data/raw/DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv
# data/raw/DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv
#
# OUTPUTS
# -------
# data/processed/03_X_train.csv       — Training features
# data/processed/03_y_train.csv       — Training labels
# data/processed/03_X_test.csv        — Test features
# data/processed/03_y_test.csv        — Test labels
# data/processed/03_feature_columns.json — Ordered list of feature names
#
# RUN ORDER
# ---------
# Run AFTER : 02_eda.ipynb
# Run BEFORE: 04_modeling.ipynb
# ============================================================

## Section 1 — Imports and Configuration

In [41]:
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

PROJECT_ROOT  = Path(os.getcwd()).parent
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root : {PROJECT_ROOT}")
print(f"Run at       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Project root : /workspaces/readmitiq
Run at       : 2026-04-09 04:36:00


## Section 2 — Load Raw Data

In [42]:
# ---------------------------------------------------------
# Load beneficiary files for 2008 and 2009.
# 2008 = training year demographics
# 2009 = test year demographics
# ---------------------------------------------------------

print("Loading beneficiary files...")

bene_2008 = pd.read_csv(RAW_DIR / "DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv", low_memory=False)
bene_2009 = pd.read_csv(RAW_DIR / "DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv", low_memory=False)

print(f"  2008 beneficiaries: {bene_2008.shape[0]:,}")
print(f"  2009 beneficiaries: {bene_2009.shape[0]:,}")

print("\nLoading inpatient claims (2008-2010)...")
ip = pd.read_csv(RAW_DIR / "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv", low_memory=False)
print(f"  Inpatient claims  : {ip.shape[0]:,}")

Loading beneficiary files...
  2008 beneficiaries: 116,352
  2009 beneficiaries: 114,538

Loading inpatient claims (2008-2010)...
  Inpatient claims  : 66,773


In [43]:
# ---------------------------------------------------------
# Parse all date columns up front.
# CMS stores dates as integers in YYYYMMDD format.
# ---------------------------------------------------------

ip["CLM_ADMSN_DT"]       = pd.to_datetime(ip["CLM_ADMSN_DT"],       format="%Y%m%d", errors="coerce")
ip["NCH_BENE_DSCHRG_DT"] = pd.to_datetime(ip["NCH_BENE_DSCHRG_DT"], format="%Y%m%d", errors="coerce")

bene_2008["BENE_BIRTH_DT"] = pd.to_datetime(bene_2008["BENE_BIRTH_DT"], format="%Y%m%d", errors="coerce")
bene_2009["BENE_BIRTH_DT"] = pd.to_datetime(bene_2009["BENE_BIRTH_DT"], format="%Y%m%d", errors="coerce")

print("Date columns parsed.")
print(f"Inpatient date range: {ip['CLM_ADMSN_DT'].min().date()} to {ip['CLM_ADMSN_DT'].max().date()}")

Date columns parsed.
Inpatient date range: 2007-11-27 to 2010-12-30


## Section 3 — Derive the 30-Day Readmission Target Variable

This is the most important step in the pipeline. The SynPUF does not include
a pre-built readmission flag — we have to engineer it from the raw claims.

**The logic:**
For each inpatient claim (the "index admission"), we look at whether the same
beneficiary was admitted again within 30 days of discharge. If yes → readmitted = 1.

In [44]:
# ---------------------------------------------------------
# Step 1: Remove records with missing or invalid dates.
# We cannot derive a readmission label without valid
# admission and discharge dates.
# ---------------------------------------------------------

n_before = len(ip)

# Drop rows where either date is null
ip = ip.dropna(subset=["CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT"])

# Drop rows where discharge is before admission (data error)
ip = ip[ip["NCH_BENE_DSCHRG_DT"] >= ip["CLM_ADMSN_DT"]]

n_after = len(ip)
print(f"Records before date cleaning : {n_before:,}")
print(f"Records after date cleaning  : {n_after:,}")
print(f"Records removed              : {n_before - n_after:,}")

Records before date cleaning : 66,773
Records after date cleaning  : 66,773
Records removed              : 0


In [45]:
# ---------------------------------------------------------
# Step 2: Sort by beneficiary and admission date.
# This ordering is critical — we need claims in chronological
# order so that shift(-1) gives us the NEXT admission.
# ---------------------------------------------------------

ip = ip.sort_values(["DESYNPUF_ID", "CLM_ADMSN_DT"]).reset_index(drop=True)

print("Claims sorted by beneficiary and admission date.")

Claims sorted by beneficiary and admission date.


In [46]:
# ---------------------------------------------------------
# Step 3: For each claim, find the next admission date
# for the same beneficiary using a grouped shift.
#
# shift(-1) moves values UP by one row within each group —
# so row N gets the admission date from row N+1 of the
# same beneficiary. The last claim for each beneficiary
# gets NaT (no next admission).
# ---------------------------------------------------------

ip["NEXT_ADMSN_DT"] = ip.groupby("DESYNPUF_ID")["CLM_ADMSN_DT"].shift(-1)

# Calculate days from discharge to next admission
ip["DAYS_TO_READMIT"] = (
    ip["NEXT_ADMSN_DT"] - ip["NCH_BENE_DSCHRG_DT"]
).dt.days

# Create the binary readmission label
# A readmission is counted if it happens 0–30 days after discharge
# (0 days = same day, which can happen for transfers)
ip["READMITTED_30"] = (
    (ip["DAYS_TO_READMIT"] >= 0) &
    (ip["DAYS_TO_READMIT"] <= 30)
).astype(int)

# Report readmission rates by year to sanity check
ip["ADMIT_YEAR"] = ip["CLM_ADMSN_DT"].dt.year
print("30-day readmission rate by year:")
print(
    ip.groupby("ADMIT_YEAR")["READMITTED_30"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "Readmit Rate", "count": "N Claims"})
    .assign(**{"Readmit Rate": lambda x: (x["Readmit Rate"] * 100).round(2)})
    .to_string()
)

30-day readmission rate by year:
            Readmit Rate  N Claims
ADMIT_YEAR                        
2007               17.26       226
2008               15.05     27701
2009                7.61     25264
2010                4.29     13582


## Section 4 — Build Features

We engineer 14 features from the raw inpatient and beneficiary data.
Each feature has a clear clinical or analytical rationale.

In [47]:
# ---------------------------------------------------------
# Feature 1: Length of Stay (LOS)
#
# We use CLM_UTLZTN_DAY_CNT — CMS's pre-calculated utilization
# day count — instead of subtracting dates ourselves.
#
# Why: CMS calculates this field per Medicare billing rules,
# accounting for covered days correctly. It is more reliable
# than date arithmetic, which can be thrown off by same-day
# admissions and discharges.
# ---------------------------------------------------------

# Use CMS's pre-calculated day count directly
ip["LOS"] = ip["CLM_UTLZTN_DAY_CNT"].fillna(0).clip(0, 180)

print(f"Feature 1 — LOS (from CLM_UTLZTN_DAY_CNT):")
print(f"  min  : {ip['LOS'].min()}")
print(f"  max  : {ip['LOS'].max()}")
print(f"  mean : {ip['LOS'].mean():.1f}")
print(f"  nulls: {ip['CLM_UTLZTN_DAY_CNT'].isnull().sum():,}")


Feature 1 — LOS (from CLM_UTLZTN_DAY_CNT):
  min  : 0.0
  max  : 136.0
  mean : 5.6
  nulls: 68


In [48]:
# ---------------------------------------------------------
# Feature 2: Claim Payment Amount (log-transformed)
# Higher-cost claims often reflect more complex cases.
# We log-transform to reduce the effect of extremely high
# outlier payments on the model.
# ---------------------------------------------------------

# Add 1 before log to handle any zero-payment claims
ip["CLM_PMT_AMT_LOG"] = np.log1p(ip["CLM_PMT_AMT"].clip(lower=0).fillna(0))

print(f"Feature 2 — CLM_PMT_AMT_LOG: min={ip['CLM_PMT_AMT_LOG'].min():.2f}, max={ip['CLM_PMT_AMT_LOG'].max():.2f}")

Feature 2 — CLM_PMT_AMT_LOG: min=0.00, max=10.95


In [49]:
# ---------------------------------------------------------
# Feature 3: Number of Diagnoses on the Claim
# Count how many ICD-9 diagnosis code fields are non-null.
# More diagnoses = more complex presentation.
# ---------------------------------------------------------

# Identify all ICD-9 diagnosis code columns in the inpatient file
icd_cols = [c for c in ip.columns if c.startswith("ICD9_DGNS_CD_")]
print(f"ICD-9 diagnosis columns found: {icd_cols}")

# Count non-null diagnosis codes per claim
ip["N_DIAGNOSES"] = ip[icd_cols].notna().sum(axis=1)

print(f"\nFeature 3 — N_DIAGNOSES: min={ip['N_DIAGNOSES'].min()}, max={ip['N_DIAGNOSES'].max()}, mean={ip['N_DIAGNOSES'].mean():.1f}")

ICD-9 diagnosis columns found: ['ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10']

Feature 3 — N_DIAGNOSES: min=0, max=10, mean=8.0


In [50]:
# ---------------------------------------------------------
# New Feature: Number of Procedure Codes
# Count non-null ICD9_PRCDR_CD_1 through _6 columns.
# More procedures on a claim = more complex surgical/clinical case,
# which is associated with higher readmission risk.
# ---------------------------------------------------------

prcdr_cols = [c for c in ip.columns if c.startswith("ICD9_PRCDR_CD_")]
print(f"Procedure code columns found: {prcdr_cols}")

ip["N_PROCEDURES"] = ip[prcdr_cols].notna().sum(axis=1)

print(f"N_PROCEDURES: min={ip['N_PROCEDURES'].min()}, "
      f"max={ip['N_PROCEDURES'].max()}, "
      f"mean={ip['N_PROCEDURES'].mean():.2f}")
print(f"Claims with zero procedures: {(ip['N_PROCEDURES']==0).sum():,} "
      f"({(ip['N_PROCEDURES']==0).mean()*100:.1f}%)")


Procedure code columns found: ['ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6']
N_PROCEDURES: min=0, max=6, mean=1.44
Claims with zero procedures: 28,506 (42.7%)


In [51]:
# ---------------------------------------------------------
# New Features: Additional Financial Complexity Signals
#
# CLM_PASS_THRU_PER_DIEM_AMT: Per diem pass-through amount.
#   A different angle on cost than total payment — captures
#   daily resource intensity of the stay.
#
# NCH_BENE_IP_DDCTBL_AMT: Inpatient deductible paid by beneficiary.
#   In Medicare Part A, the deductible resets each benefit period.
#   A non-zero deductible signals a new benefit period, which can
#   indicate a gap since the last hospitalization.
#
# HAS_OTHER_PAYER: Binary flag — is NCH_PRMRY_PYR_CLM_PD_AMT > 0?
#   If Medicare was NOT the primary payer, another insurer paid first.
#   This is a proxy for dual-eligible status (Medicare + Medicaid),
#   which is one of the strongest predictors of readmission.
# ---------------------------------------------------------

# Per diem — log-transform like we did for total payment
ip["PER_DIEM_LOG"] = np.log1p(
    ip["CLM_PASS_THRU_PER_DIEM_AMT"].clip(lower=0).fillna(0)
)

# Deductible amount — log-transform
ip["DDCTBL_AMT_LOG"] = np.log1p(
    ip["NCH_BENE_IP_DDCTBL_AMT"].clip(lower=0).fillna(0)
)

# Other payer flag — 1 if Medicare was NOT the primary payer
ip["HAS_OTHER_PAYER"] = (
    ip["NCH_PRMRY_PYR_CLM_PD_AMT"].fillna(0) > 0
).astype(int)

print(f"PER_DIEM_LOG   : mean={ip['PER_DIEM_LOG'].mean():.2f}, max={ip['PER_DIEM_LOG'].max():.2f}")
print(f"DDCTBL_AMT_LOG : mean={ip['DDCTBL_AMT_LOG'].mean():.2f}, max={ip['DDCTBL_AMT_LOG'].max():.2f}")
print(f"HAS_OTHER_PAYER: {ip['HAS_OTHER_PAYER'].mean()*100:.1f}% of claims have another primary payer")


PER_DIEM_LOG   : mean=1.16, max=6.22
DDCTBL_AMT_LOG : mean=6.74, max=7.00
HAS_OTHER_PAYER: 2.3% of claims have another primary payer


In [52]:
# ---------------------------------------------------------
# Feature 4: DRG Major Diagnostic Category (MDC)
#
# DRG codes group patients by clinical condition and resource use.
# We map DRG to MDC (broader category) to reduce cardinality.
#
# MDC ranges are based on CMS's official DRG-to-MDC crosswalk.
# Reference: https://www.cms.gov/icd10m/FY2024-version41-fullcode-cms
# ---------------------------------------------------------

def drg_to_mdc(drg_code):
    """
    Map a DRG code to its Major Diagnostic Category (MDC).
    Returns an integer MDC 1-25, or 0 for pre-MDC / unknown.

    This mapping covers the most common MDC ranges.
    Full production version would use the CMS DRG definition manual.
    """
    try:
        drg = int(drg_code)
    except (ValueError, TypeError):
        return 0

    # Pre-MDC (transplants, tracheostomies) — high complexity
    if 1 <= drg <= 17:
        return 0
    # MDC 1: Nervous System
    elif 20 <= drg <= 103:
        return 1
    # MDC 2: Eye
    elif 113 <= drg <= 125:
        return 2
    # MDC 3: Ear, Nose, Mouth, Throat
    elif 129 <= drg <= 159:
        return 3
    # MDC 4: Respiratory
    elif 163 <= drg <= 208:
        return 4
    # MDC 5: Circulatory (includes heart failure, AMI)
    elif 215 <= drg <= 316:
        return 5
    # MDC 6: Digestive
    elif 326 <= drg <= 395:
        return 6
    # MDC 7: Hepatobiliary
    elif 405 <= drg <= 446:
        return 7
    # MDC 8: Musculoskeletal
    elif 453 <= drg <= 566:
        return 8
    # MDC 9: Skin
    elif 573 <= drg <= 607:
        return 9
    # MDC 10: Endocrine (includes diabetes)
    elif 614 <= drg <= 645:
        return 10
    # MDC 11: Kidney and Urinary Tract
    elif 652 <= drg <= 700:
        return 11
    # MDC 19-25: Mental health, substance use, etc.
    elif 876 <= drg <= 923:
        return 19
    else:
        return 0  # Unknown / other

ip["DRG_MDC"] = ip["CLM_DRG_CD"].apply(drg_to_mdc)

print(f"Feature 4 — DRG_MDC distribution:")
print(ip["DRG_MDC"].value_counts().head(10).to_string())

Feature 4 — DRG_MDC distribution:
DRG_MDC
5     14850
4      9407
0      9038
8      7793
6      6681
1      5119
11     4135
19     3493
10     2400
7      1726


In [53]:
# ---------------------------------------------------------
# Feature 5: Prior Inpatient Count (12-month lookback)
#
# Patients with frequent prior hospitalizations are at
# substantially higher readmission risk.
#
# For each admission, we count how many OTHER inpatient
# claims that beneficiary had in the 12 months prior.
# ---------------------------------------------------------

print("Calculating prior inpatient utilization (this may take a moment)...")

# We need the full multi-year claims dataset for this lookback
# Sort by beneficiary and date
ip_for_lookback = ip[["DESYNPUF_ID", "CLM_ADMSN_DT"]].copy()

prior_counts = []

# Group by beneficiary to avoid cross-beneficiary comparisons
for bene_id, group in ip_for_lookback.groupby("DESYNPUF_ID"):
    dates = group["CLM_ADMSN_DT"].sort_values().tolist()
    for i, admit_dt in enumerate(dates):
        # Count admissions strictly before this one and within 365 days
        lookback_start = admit_dt - pd.Timedelta(days=365)
        prior = sum(
            1 for j, d in enumerate(dates)
            if j != i and lookback_start <= d < admit_dt
        )
        prior_counts.append(prior)

ip["PRIOR_INPATIENT_CNT"] = prior_counts

print(f"Feature 5 — PRIOR_INPATIENT_CNT: mean={ip['PRIOR_INPATIENT_CNT'].mean():.2f}, max={ip['PRIOR_INPATIENT_CNT'].max()}")

Calculating prior inpatient utilization (this may take a moment)...
Feature 5 — PRIOR_INPATIENT_CNT: mean=0.66, max=10


In [54]:
# ---------------------------------------------------------
# Features 6-14: Join beneficiary demographics and
# chronic condition flags to the inpatient claims.
#
# IMPORTANT — Test Set Restriction (Right-Censoring Fix):
# The SynPUF 2010 data is sparse due to beneficiary attrition.
# 2009 admissions whose 30-day readmission window extends into
# 2010 are often missing their readmission records, which
# artificially deflates the 2009 readmission rate (~7.6% vs ~15%).
#
# Fix: Restrict test set to Jan–Jun 2009 admissions only.
# Any patient discharged by July 31, 2009 has their full
# 30-day window entirely within 2009 data. No 2010 dependency.
# This gives a consistent, comparable base rate in both sets.
# ---------------------------------------------------------

# Define the demographic and condition columns we need from the bene file
BENE_FEATURE_COLS = [
    "DESYNPUF_ID",
    "BENE_BIRTH_DT",
    "BENE_SEX_IDENT_CD",
    "BENE_RACE_CD",
    "SP_CHF",
    "SP_DIABETES",
    "SP_COPD",
    "SP_CHRNKIDN",
    "SP_STRKETIA",
    "SP_ALZHDMTA",
    "SP_DEPRESSN",
    "SP_ISCHMCHT",
    "SP_OSTEOPRS",
    "SP_RA_OA",
    "SP_CNCR",
]

# Filter bene files to only the columns we need
bene_2008_slim = bene_2008[BENE_FEATURE_COLS].copy()
bene_2009_slim = bene_2009[BENE_FEATURE_COLS].copy()

# Training set: all 2008 admissions
# Their 30-day windows fall into 2009, which is a full year of data
ip_2008 = ip[ip["ADMIT_YEAR"] == 2008].copy()

# Test set: Jan-Jun 2009 admissions ONLY
# Restricting to H1 2009 ensures discharge dates fall before Aug 1 at latest,
# so the full 30-day readmission window is observable within 2009 data.
# This eliminates dependence on sparse 2010 SynPUF data.
ip_2009_all  = ip[ip["ADMIT_YEAR"] == 2009].copy()
ip_2009      = ip_2009_all[ip_2009_all["CLM_ADMSN_DT"].dt.month <= 6].copy()

print(f"Training set (2008 all)           : {len(ip_2008):,} admissions")
print(f"Test set (2009 Jan-Jun only)       : {len(ip_2009):,} admissions")
print(f"2009 admissions excluded (Jul-Dec) : {len(ip_2009_all) - len(ip_2009):,}")
print()
print(f"Train readmission rate: {ip_2008['READMITTED_30'].mean()*100:.1f}%")
print(f"Test  readmission rate: {ip_2009['READMITTED_30'].mean()*100:.1f}%")

# Join each year's claims to the matching beneficiary year
df_2008 = ip_2008.merge(bene_2008_slim, on="DESYNPUF_ID", how="inner")
df_2009 = ip_2009.merge(bene_2009_slim, on="DESYNPUF_ID", how="inner")

print(f"\n2008 claims after bene join: {len(df_2008):,}")
print(f"2009 claims after bene join: {len(df_2009):,}")


Training set (2008 all)           : 27,701 admissions
Test set (2009 Jan-Jun only)       : 13,098 admissions
2009 admissions excluded (Jul-Dec) : 12,166

Train readmission rate: 15.1%
Test  readmission rate: 8.7%

2008 claims after bene join: 27,701
2009 claims after bene join: 13,098


In [55]:
# ---------------------------------------------------------
# Feature 6: Age at Admission
# Calculated from beneficiary birth date and the
# admission date of each claim.
# ---------------------------------------------------------

def calc_age(df):
    """Calculate age in years at time of admission."""
    df["AGE_AT_ADMISSION"] = (
        (df["CLM_ADMSN_DT"] - df["BENE_BIRTH_DT"]).dt.days / 365.25
    ).round(0).astype("Int64")
    # Clip to reasonable range — SynPUF occasionally has noise
    df["AGE_AT_ADMISSION"] = df["AGE_AT_ADMISSION"].clip(0, 110)
    return df

df_2008 = calc_age(df_2008)
df_2009 = calc_age(df_2009)

print(f"Feature 6 — AGE_AT_ADMISSION (2008): mean={df_2008['AGE_AT_ADMISSION'].mean():.1f}")

Feature 6 — AGE_AT_ADMISSION (2008): mean=73.6


In [56]:
# ---------------------------------------------------------
# Feature 7: Number of Chronic Conditions
# Sum all SP_* flag columns to get total comorbidity burden.
# This is a strong readmission predictor as shown in EDA.
# ---------------------------------------------------------

SP_COLS = [c for c in df_2008.columns if c.startswith("SP_")]

def add_chronic_count(df):
    """Sum all chronic condition flags into a single count."""
    df["N_CHRONIC_CONDITIONS"] = df[SP_COLS].fillna(0).sum(axis=1)
    return df

df_2008 = add_chronic_count(df_2008)
df_2009 = add_chronic_count(df_2009)

print(f"Chronic condition columns: {SP_COLS}")
print(f"\nFeature 7 — N_CHRONIC_CONDITIONS (2008): mean={df_2008['N_CHRONIC_CONDITIONS'].mean():.2f}")

Chronic condition columns: ['SP_CHF', 'SP_DIABETES', 'SP_COPD', 'SP_CHRNKIDN', 'SP_STRKETIA', 'SP_ALZHDMTA', 'SP_DEPRESSN', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_CNCR']

Feature 7 — N_CHRONIC_CONDITIONS (2008): mean=15.84


## Section 5 — Assemble Final Feature Matrix

In [57]:
# ---------------------------------------------------------
# Define the final ordered list of feature columns.
# This order MUST be saved and used consistently in the
# training script and inference API — any mismatch will
# cause the model to produce wrong predictions.
# ---------------------------------------------------------

FEATURE_COLUMNS = [
    # Claim-level features
    "LOS",                   # Length of stay (days, from CLM_UTLZTN_DAY_CNT)
    "CLM_PMT_AMT_LOG",       # Log of total claim payment
    "PER_DIEM_LOG",          # Log of per diem pass-through amount
    "DDCTBL_AMT_LOG",        # Log of inpatient deductible paid
    "HAS_OTHER_PAYER",       # 1 if Medicare was not primary payer (dual eligible proxy)
    "N_DIAGNOSES",           # Count of ICD-9 diagnosis codes on claim
    "N_PROCEDURES",          # Count of ICD-9 procedure codes on claim
    "DRG_MDC",               # Major Diagnostic Category
    # Patient demographics
    "AGE_AT_ADMISSION",      # Age in years at time of admission
    "BENE_SEX_IDENT_CD",     # Sex (1=Male, 2=Female)
    "BENE_RACE_CD",          # Race/ethnicity code
    # Chronic condition flags
    "N_CHRONIC_CONDITIONS",  # Total chronic condition count
    "SP_CHF",                # Heart Failure
    "SP_DIABETES",           # Diabetes
    "SP_COPD",               # COPD
    "SP_CHRNKIDN",           # Chronic Kidney Disease
    "SP_STRKETIA",           # Stroke / TIA
    # Prior utilization
    "PRIOR_INPATIENT_CNT",   # Prior 12-month inpatient admissions
]

TARGET_COLUMN = "READMITTED_30"

print(f"Feature count : {len(FEATURE_COLUMNS)} (was 14, now expanded)")
for f in FEATURE_COLUMNS:
    print(f"  {f}")


Feature count : 18 (was 14, now expanded)
  LOS
  CLM_PMT_AMT_LOG
  PER_DIEM_LOG
  DDCTBL_AMT_LOG
  HAS_OTHER_PAYER
  N_DIAGNOSES
  N_PROCEDURES
  DRG_MDC
  AGE_AT_ADMISSION
  BENE_SEX_IDENT_CD
  BENE_RACE_CD
  N_CHRONIC_CONDITIONS
  SP_CHF
  SP_DIABETES
  SP_COPD
  SP_CHRNKIDN
  SP_STRKETIA
  PRIOR_INPATIENT_CNT


In [58]:
# ---------------------------------------------------------
# Verify all feature columns exist in both dataframes
# before trying to extract them.
# ---------------------------------------------------------

missing_2008 = [c for c in FEATURE_COLUMNS if c not in df_2008.columns]
missing_2009 = [c for c in FEATURE_COLUMNS if c not in df_2009.columns]

if missing_2008:
    raise ValueError(f"Missing in 2008 data: {missing_2008}")
if missing_2009:
    raise ValueError(f"Missing in 2009 data: {missing_2009}")

print("✅ All feature columns present in both 2008 and 2009 data.")

✅ All feature columns present in both 2008 and 2009 data.


In [59]:
# ---------------------------------------------------------
# Extract feature matrices and target vectors.
# 2008 = training set  (model learns from this)
# 2009 = test set      (we evaluate on this — never seen during training)
#
# This is a TEMPORAL split — not random.
# We train on the past and evaluate on the future.
# This is the correct approach for time-series healthcare data.
# ---------------------------------------------------------

# Fill any remaining nulls in feature columns with 0
# (SP_ flags default to 0 in SynPUF; other nulls are rare)
X_train = df_2008[FEATURE_COLUMNS].fillna(0).reset_index(drop=True)
y_train = df_2008[TARGET_COLUMN].reset_index(drop=True)

X_test  = df_2009[FEATURE_COLUMNS].fillna(0).reset_index(drop=True)
y_test  = df_2009[TARGET_COLUMN].reset_index(drop=True)

print(f"Training set : {X_train.shape[0]:,} samples | {X_train.shape[1]} features")
print(f"  Readmission rate: {y_train.mean()*100:.1f}%")
print()
print(f"Test set     : {X_test.shape[0]:,} samples  | {X_test.shape[1]} features")
print(f"  Readmission rate: {y_test.mean()*100:.1f}%")

Training set : 27,701 samples | 18 features
  Readmission rate: 15.1%

Test set     : 13,098 samples  | 18 features
  Readmission rate: 8.7%


In [60]:
# ---------------------------------------------------------
# Final check: look for any remaining null values.
# The model will fail to train if there are NaN values.
# ---------------------------------------------------------

train_nulls = X_train.isnull().sum().sum()
test_nulls  = X_test.isnull().sum().sum()

print(f"Null values in X_train : {train_nulls}")
print(f"Null values in X_test  : {test_nulls}")

if train_nulls > 0 or test_nulls > 0:
    print("\n⚠️  Nulls found — check the columns above before proceeding.")
else:
    print("\n✅ No null values. Ready for modeling.")

Null values in X_train : 0
Null values in X_test  : 0

✅ No null values. Ready for modeling.


## Section 6 — Save Outputs

In [61]:
# ---------------------------------------------------------
# Save feature matrices and labels to data/processed/.
# All files prefixed with 03_ to match this notebook.
# ---------------------------------------------------------

X_train.to_csv(PROCESSED_DIR / "03_X_train.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "03_y_train.csv", index=False, header=True)
X_test.to_csv(PROCESSED_DIR  / "03_X_test.csv",  index=False)
y_test.to_csv(PROCESSED_DIR  / "03_y_test.csv",  index=False, header=True)

# Save the feature column order as JSON.
# The API and training script MUST use this file to ensure
# feature order is always consistent with what the model expects.
with open(PROCESSED_DIR / "03_feature_columns.json", "w") as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print("Saved:")
print(f"  03_X_train.csv       ({X_train.shape[0]:,} rows x {X_train.shape[1]} cols)")
print(f"  03_y_train.csv       ({y_train.shape[0]:,} rows)")
print(f"  03_X_test.csv        ({X_test.shape[0]:,} rows x {X_test.shape[1]} cols)")
print(f"  03_y_test.csv        ({y_test.shape[0]:,} rows)")
print(f"  03_feature_columns.json ({len(FEATURE_COLUMNS)} features)")

Saved:
  03_X_train.csv       (27,701 rows x 18 cols)
  03_y_train.csv       (27,701 rows)
  03_X_test.csv        (13,098 rows x 18 cols)
  03_y_test.csv        (13,098 rows)
  03_feature_columns.json (18 features)


In [62]:
# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("=" * 60)
print("NOTEBOOK 03 — COMPLETE")
print("=" * 60)
print()
print(f"  Training samples : {X_train.shape[0]:,}  (2008 admissions)")
print(f"  Test samples     : {X_test.shape[0]:,}  (2009 admissions)")
print(f"  Features         : {len(FEATURE_COLUMNS)}")
print(f"  Train readmit %  : {y_train.mean()*100:.1f}%")
print(f"  Test readmit %   : {y_test.mean()*100:.1f}%")
print()
print("Next step:")
print("  Open 04_modeling.ipynb to train and evaluate the XGBoost model.")
print("=" * 60)

NOTEBOOK 03 — COMPLETE

  Training samples : 27,701  (2008 admissions)
  Test samples     : 13,098  (2009 admissions)
  Features         : 18
  Train readmit %  : 15.1%
  Test readmit %   : 8.7%

Next step:
  Open 04_modeling.ipynb to train and evaluate the XGBoost model.
